# 1. Backtest a stock-selection strategy

Suppose we have $1,000,000 to invest, cannot borrow, and must hold each purchase for 90 calendar days. Every month, the model supplies a new ranking of stocks. A good forecast alone does not tell us how much we can buy: earlier purchases may still tie up most of our money.

Each purchase creates a **lot**, with its own number of units, entry date, and scheduled exit. Purchases made on the same monthly entry date form a **cohort**. January's cohort may still be held when February's and March's cohorts enter. Buying the same stock in two months creates two lots that mature on different dates.

We'll work through one purchase and sale using AAPL prices, then track overlapping lots, cash, fees, and daily portfolio value. HW4 asks you to design your own strategy and backtester for the full dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import sys
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
modeling_dir = course_root / 'Lectures/modeling'
if str(modeling_dir) not in sys.path:
    sys.path.insert(0, str(modeling_dir))
from course_data import (load_membership, apply_membership, load_data, prepare_panel, price_features, financial_events,
                        join_financials, make_labels, date_folds, make_pipeline,
                        prediction_metrics, QuantileClipper, PRICE_FEATURES,
                        FINANCIAL_FEATURES, NUMERIC_FEATURES, FEATURES)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

from backtest import run_backtest, performance
from backtest import HeldPriceGapError

In [ ]:
prices, fundamentals, universe, benchmark = load_data()
calendar = pd.DatetimeIndex(benchmark.date)
panel = prepare_panel()
display(panel.loc[panel.date.eq('2012-12-31'), ['symbol', 'momentum_63', 'eligible']].head())

## 1. Decide what happens on each date

A backtest must distinguish the time we decide to trade from the time we receive a fill price. Here we decide after one market close and execute at the next close. The dollar budget is known when we make the decision; the number of units is calculated from the later execution price.

| When | What happens | Available information |
|---|---|---|
| After signal close `t` | Build features, fit if scheduled, rank stocks, set dollar budgets | Observations and completed labels through `t` |
| Next session close `e` | Execute the planned purchases | The execution price determines units; the stock choices are already fixed |
| Each later session | Value every open lot and record NAV | Today's observed prices, or a flagged stale mark |
| First quoted session on/after `e + 90 calendar days` | Sell the lot and charge exit cost | Its predetermined maturity and today's quote |

Purchase budgets use cash carried from the previous close. If a lot sells today, its proceeds can fund tomorrow's orders. They cannot enlarge an order decided yesterday, because yesterday we did not know today's sale proceeds. The loop must preserve this distinction even if it processes sales before purchases.

We use adjusted close for entry, daily valuation, and exit, allowing fractional units. Adjusted prices approximate a total-return investment by incorporating the vendor's adjustments for distributions and corporate actions. Crediting dividends separately would therefore count them twice. These units are not actual historical share counts; modeling actual shares would require raw prices, splits, dividend cash, and a more detailed execution model. The 5-basis-point cost on each side is an assumption, not an estimate of market impact.

Quarterly membership decides which stocks may receive new purchases. It does not cancel an older purchase's 90-day holding period. Keep prices for stocks that leave the universe so their outstanding lots can still be valued and sold.

## 2. Follow one AAPL purchase through to its sale

Use the signal after December 31, 2012. The next market session is January 2, 2013, and the 90-calendar-day maturity is April 2. The extract's adjusted AAPL prices on those dates are $16.58 and $13.05.

The purchase has a **$100,000 budget including its entry fee**. Five basis points means a cost rate of `0.0005`. If we spent the full $100,000 on the position and charged the fee afterward, we would exceed the budget. Instead, solve for the units $q$ using budget $B$, price $P$, and fee rate $c$:

$$qP + cqP = B, \qquad q = \frac{B}{P(1+c)}.$$

This buys about $99,950.02 of exposure and pays a $49.98 fee. Cash falls from $1,000,000 to $900,000. The new position is worth $99,950.02, so total portfolio value immediately falls to $999,950.02: the fee is already a loss even though the market price has not moved.

On exit, sell the units at the observed price and deduct another fee from the proceeds. The table below records both cash movements. Check that their sum equals the lot's net profit or loss.

In [ ]:
signal_date = pd.Timestamp('2012-12-31')
entry_date = calendar[calendar.searchsorted(signal_date, side='right')]
due_date = entry_date + pd.Timedelta(days=90)
aapl = prices.loc[prices.symbol.eq('AAPL')].set_index('date').adj_close.sort_index()
exit_date = aapl.index[aapl.index.searchsorted(due_date)]
budget, fee_rate = 100_000., 5 / 10_000
entry_price, exit_price = aapl.loc[entry_date], aapl.loc[exit_date]
notional = budget / (1 + fee_rate)
units = notional / entry_price
entry_fee = notional * fee_rate
exit_notional = units * exit_price
exit_fee = exit_notional * fee_rate
cash = 1_000_000 - budget
nav_after_entry = cash + units * entry_price
cash_after_exit = cash + exit_notional - exit_fee
lot_trace = pd.DataFrame([
    {'date': entry_date, 'event': 'BUY', 'price': entry_price, 'units': units,
     'fee': entry_fee, 'cash_change': -budget},
    {'date': exit_date, 'event': 'SELL', 'price': exit_price, 'units': units,
     'fee': exit_fee, 'cash_change': exit_notional - exit_fee},
])
display(lot_trace)
display(pd.Series({'due_date': due_date, 'actual_exit': exit_date,
                   'NAV_after_entry': nav_after_entry, 'cash_after_exit': cash_after_exit,
                   'net_lot_PnL': exit_notional - exit_fee - budget}))
assert np.isclose(nav_after_entry, 1_000_000 - entry_fee)
assert np.isclose(cash_after_exit, 1_000_000 + lot_trace.cash_change.sum())

## 3. Include open positions in daily portfolio value

Between entry and exit, the AAPL lot is still owned. A falling price reduces our wealth even though no sale has happened. Reporting only cash or completed trades would hide that loss until the exit date.

**Net asset value (NAV)** is cash plus the current market value of all open lots:

$$NAV_d = cash_d + \sum_{\ell\text{ open}} q_\ell P_{\ell,d}.$$

Here $q_\ell$ is the number of units in lot $\ell$, and $P_{\ell,d}$ is its price on date $d$. In the single-lot example, cash stays at $900,000 until the sale; all interim NAV changes come from AAPL's price. At exit, its units become zero and the net sale proceeds move into cash. The position must not remain in holdings after those proceeds have been credited, or we would count its value twice.

The chart below values the lot at every close. Later, when a stock is bought repeatedly, keep a separate lot ID for each purchase so it exits on its own due date. Add the lots together when calculating the total exposure to that stock.

In [ ]:
lot_path = aapl.loc[entry_date:exit_date].to_frame('adjusted_price')
lot_path['units'] = units
lot_path['cash'] = cash
lot_path['nav'] = lot_path.cash + lot_path.units * lot_path.adjusted_price
lot_path.loc[exit_date, ['units', 'cash', 'nav']] = [0, cash_after_exit, cash_after_exit]
lot_path.nav.plot(title='AAPL lot plus idle cash: mark every close')
plt.ylabel('Portfolio value ($)')
plt.tight_layout()
display(lot_path.iloc[[0, 1, -2, -1]])

## 4. Give the backtester scores and record what it does

The model's output is a table with **`date, symbol, score`**: on a given signal date, how highly does it rank each eligible stock? The backtester turns that ranking into orders, fills, holdings, and cash. It receives the price history for simulation, but each allocation decision may use only information already available at its signal date. Targets and future exit dates are excluded from the score table.

The following outputs answer different accounting questions:

| Table | Main columns | What we can check |
|---|---|---|
| Fills | date, lot ID, side, units, price, notional, fee, cash change, due date | What actually traded, and how much cash moved? |
| Open lots | lot ID, symbol, entry date, units, due date | What do we still own? |
| Daily history | cash, holdings value, NAV, fees, gross traded, open lots, stale marks | What was the portfolio worth each day? |
| Diagnostics | date, symbol, event | Which orders or prices needed special handling? |

Our class strategy buys the top three scores on the first market session of each month. Its total purchase budget is the smaller of one-third of previous-close NAV and the cash available at that close. Thus a strong new signal cannot make the strategy borrow to buy more. If a chosen stock has no execution quote, reject its order and leave that money in cash; choosing a replacement using execution-day information would change the decision rule.

A missing quote on a holding is harder: the shares have not disappeared just because the data has. Carry the last observed mark for at most five market sessions, flagging those valuations as stale. A longer gap stops the run for investigation. The final row in a price file does not establish when a company delisted or what its shareholders received, so it cannot justify inventing a sale at that price.

### Check portfolio limits before allocating each order

A 5% name cap limits the combined position in a stock, not each individual purchase. If we already own several AAPL lots, add their values before deciding whether another purchase fits. A sector cap similarly combines all stocks assigned to that sector.

`capped_budgets` starts with holdings valued at the previous close and calculates the dollars still available under each limit. The proposed purchase receives the smallest budget allowed by equal allocation, cash, the name cap, and the sector cap. Fees fit inside that budget; only the position's notional value uses up exposure capacity.

After planning an order, immediately add its notional to the planned name and sector totals. Otherwise two technology purchases could each appear to fit into the same remaining sector capacity while exceeding it together. Any unused budget stays in cash.

Market moves may already have pushed an existing position above a cap. It receives no additional purchases, but the 90-day rule still prevents an early sale. Also, execution-day prices can change realized weights after the prior-close allocation check. Record these breaches rather than assuming a planned cap guarantees every later weight.

The reference loop below does not call `capped_budgets`. You can add limits like these if you want to control concentration in your strategy.

In [ ]:
def capped_budgets(scores, holdings, sectors, cash, nav, top_n=20,
                   cohort_fraction=1 / 3, name_cap=.05, sector_cap=.25, fee_bps=5):
    """Plan inclusive dollar budgets from *prior-close* data, before execution.

    holdings is aggregate market value by ticker. Scores and sectors are also
    ticker-indexed Series. Reduce each equal budget as needed, retaining unused
    cash. Update planned exposures after every allocation, not after the loop.
    """
    if not scores.index.is_unique or not holdings.index.is_unique or not sectors.index.is_unique:
        raise ValueError("Aggregate duplicate tickers before allocating")
    if nav <= 0 or cash < 0 or not np.isclose(cash + holdings.sum(), nav):
        raise ValueError("Prior-close holdings and cash must reconcile to NAV")
    if not np.isfinite(scores).all() or not np.isfinite(holdings).all() or holdings.lt(0).any():
        raise ValueError("Scores/holdings must be finite; holdings nonnegative")
    if top_n < 1 or fee_bps < 0 or not all(0 < cap <= 1 for cap in [cohort_fraction, name_cap, sector_cap]):
        raise ValueError("Invalid allocation setting")
    sectors = sectors.fillna("Unknown")
    if not set(scores.index).union(holdings.index).issubset(sectors.index):
        raise ValueError("Every scored/held ticker needs an explicit sector bucket")
    ranked = scores.rename_axis("symbol").rename("score").reset_index().sort_values(
        ["score", "symbol"], ascending=[False, True]).head(top_n)
    fee = fee_bps / 10_000
    budget = min(cash, nav * cohort_fraction)
    equal_budget = budget / len(ranked) if len(ranked) else 0.
    planned_names = holdings.to_dict()
    planned_sectors = holdings.groupby(sectors.reindex(holdings.index)).sum().to_dict()
    orders = {}
    for symbol in ranked.symbol:
        sector = sectors.loc[symbol]
        name_room = max(0., name_cap * nav - planned_names.get(symbol, 0.))
        sector_room = max(0., sector_cap * nav - planned_sectors.get(sector, 0.))
        inclusive = min(equal_budget, budget, name_room * (1 + fee), sector_room * (1 + fee))
        notional = inclusive / (1 + fee)
        if inclusive > 1e-8:
            orders[symbol] = inclusive
            budget -= inclusive
            planned_names[symbol] = planned_names.get(symbol, 0.) + notional
            planned_sectors[sector] = planned_sectors.get(sector, 0.) + notional
    return pd.Series(orders, dtype=float, name="inclusive_budget")

Use the AAPL lot above to plan another purchase from its entry-day close. The existing holding already exceeds the 5% name cap, so `capped_budgets` blocks additional AAPL purchases. It leaves the lot in place until its scheduled maturity.

In [ ]:
prior_holdings = pd.Series({'AAPL': units * entry_price})
prior_cash = cash
prior_nav = prior_cash + prior_holdings.sum()
planning_scores = panel.loc[panel.date.eq(entry_date) & panel.eligible].set_index('symbol').momentum_63
sector_map = universe.set_index('symbol').sector
planned = capped_budgets(planning_scores, prior_holdings, sector_map, prior_cash, prior_nav,
                         top_n=20, name_cap=.05, sector_cap=.25, fee_bps=5)
planned_notionals = planned / 1.0005
post_plan = prior_holdings.add(planned_notionals, fill_value=0)
display(pd.DataFrame({'prior_value': prior_holdings, 'inclusive_budget': planned,
                      'planned_value': post_plan, 'planned_weight': post_plan/prior_nav}).fillna(0))
display((post_plan.groupby(sector_map.reindex(post_plan.index)).sum()/prior_nav).rename('planned_sector_weight'))
assert 'AAPL' not in planned.index
assert planned.sum() <= prior_cash + 1e-8
assert (planned_notionals <= .05*prior_nav + 1e-8).all()

### Read the loop as one day's accounting

For each session, the loop first saves yesterday's cash balance, then updates observed prices. It sells lots that have reached maturity and have a quote, recording their exit fees and cash proceeds. It then executes the purchases already planned from the previous close. Saving the earlier cash balance prevents today's sales from expanding those purchases.

After trading, it adds cash and the marked value of every remaining lot to produce NAV. `lots.copy()` allows the code to remove sold lots while iterating without skipping the next item in the list.

The first history row records the original capital before any transaction. Without it, calculating returns from the first post-trade NAV would miss the initial entry fees. The strategy and S&P 500 total-return benchmark start investing at the first test session's close; the benchmark has no return over that initial interval.

At the end of the test, some 90-day lots are still open. Keep them in NAV at the final available mark. Looking beyond the test end for their exit prices would use later data, while selling them early would change the holding rule.

In [ ]:
def run_backtest(prices, benchmark, scores, start, end, initial_cash=1_000_000,
                 top_n=3, cohort_fraction=1 / 3, fee_bps=5, holding_days=90):
    """Use yesterday's scores; buy monthly with carried cash; mark every session."""
    if initial_cash <= 0 or top_n < 1 or not 0 < cohort_fraction <= 1 or fee_bps < 0:
        raise ValueError("Invalid capital, selection, budget, or fee parameter")
    if prices.duplicated(["symbol", "date"]).any() or scores.duplicated(["symbol", "date"]).any():
        raise ValueError("Duplicate price or prediction key")
    if not np.isfinite(scores.score).all():
        raise ValueError("Nonfinite prediction")
    if not (np.isfinite(prices.adj_close) & prices.adj_close.gt(0)).all():
        raise ValueError("Invalid price")
    calendar = pd.DatetimeIndex(benchmark.date).sort_values()
    if calendar.has_duplicates or not (np.isfinite(benchmark.adj_close) & benchmark.adj_close.gt(0)).all():
        raise ValueError("Invalid benchmark")
    days = calendar[(calendar >= pd.Timestamp(start)) & (calendar <= pd.Timestamp(end))]
    if not len(days) or calendar.get_loc(days[0]) == 0:
        raise ValueError("Need test sessions and one preceding signal session")
    quotes = prices.pivot(index="date", columns="symbol", values="adj_close").reindex(calendar)
    score_groups = {date: group for date, group in scores.groupby("date")}
    fee = fee_bps / 10_000
    cash = float(initial_cash)
    lots, fills, diagnostics = [], [], []
    previous_day = calendar[calendar.get_loc(days[0]) - 1]
    history = [{"date": previous_day, "cash": cash, "holdings_value": 0., "nav": cash,
                "open_lots": 0, "fees": 0., "gross_traded": 0., "stale_marks": 0}]
    last_marks, last_seen = {}, {}
    next_lot_id = 0
    for day in days:
        day_index = calendar.get_loc(day)
        signal_date = calendar[day_index - 1]
        carried_cash = cash  # Today's unknown sale proceeds cannot fund precommitted buys.
        daily_fees, gross_traded, stale_marks = 0., 0., 0
        observed = quotes.loc[day].dropna()
        for symbol, price in observed.items():
            last_marks[symbol], last_seen[symbol] = price, day_index
        for lot in lots.copy():
            symbol = lot["symbol"]
            if day_index - last_seen[symbol] > 5:
                raise HeldPriceGapError(symbol, day, calendar[last_seen[symbol]])
            if symbol not in observed:
                stale_marks += 1
                diagnostics.append({"date": day, "symbol": symbol, "event": "stale_mark"})
            if day >= lot["due_date"] and symbol in observed:
                notional = lot["units"] * observed[symbol]
                cost = notional * fee
                cash += notional - cost
                daily_fees += cost
                gross_traded += notional
                fills.append({"date": day, "signal_date": lot["signal_date"], "lot_id": lot["lot_id"],
                              "symbol": symbol, "side": "SELL", "price": observed[symbol],
                              "units": lot["units"], "notional": notional, "fee": cost,
                              "cash_change": notional - cost, "due_date": lot["due_date"]})
                lots.remove(lot)
        is_entry_day = day == days[0] or day.to_period("M") != signal_date.to_period("M")
        if is_entry_day and signal_date in score_groups:
            # Rankings and total budget are fixed using the previous close's information.
            ranked = score_groups[signal_date].sort_values(["score", "symbol"], ascending=[False, True])
            chosen = ranked.head(top_n)
            budget = min(carried_cash, history[-1]["nav"] * cohort_fraction)
            per_name = budget / len(chosen) if len(chosen) else 0
            for row in chosen.itertuples():
                if row.symbol not in observed:
                    diagnostics.append({"date": day, "symbol": row.symbol, "event": "rejected_missing_quote"})
                    continue  # Do not substitute another name using execution-day information.
                if per_name <= 1e-8:
                    continue
                notional = per_name / (1 + fee)  # Reserve the entry fee inside the cash budget.
                cost = notional * fee
                units = notional / observed[row.symbol]
                cash -= notional + cost
                daily_fees += cost
                gross_traded += notional
                lot = {"lot_id": next_lot_id, "symbol": row.symbol, "units": units,
                       "signal_date": signal_date, "entry_date": day,
                       "entry_price": observed[row.symbol], "due_date": day + pd.Timedelta(days=holding_days)}
                lots.append(lot)
                next_lot_id += 1
                fills.append({"date": day, "signal_date": signal_date, "lot_id": lot["lot_id"],
                              "symbol": row.symbol, "side": "BUY", "price": observed[row.symbol],
                              "units": units, "notional": notional, "fee": cost,
                              "cash_change": -(notional + cost), "due_date": lot["due_date"]})
        holdings = sum(lot["units"] * last_marks[lot["symbol"]] for lot in lots)
        if cash < -1e-7:
            raise AssertionError("Cash became negative")
        history.append({"date": day, "cash": cash, "holdings_value": holdings, "nav": cash + holdings,
                        "open_lots": len(lots), "fees": daily_fees, "gross_traded": gross_traded,
                        "stale_marks": stale_marks})
    history = pd.DataFrame(history).set_index("date")
    b = benchmark.set_index("date").adj_close.reindex(days)
    history["benchmark_nav"] = float(initial_cash)
    history.loc[days, "benchmark_nav"] = initial_cash * b / b.iloc[0]
    # Initial row is pre-entry capital. First day's return includes all entry costs.
    return {"history": history,
            "fills": pd.DataFrame(fills, columns=["date", "signal_date", "lot_id", "symbol", "side", "price",
                                                 "units", "notional", "fee", "cash_change", "due_date"]),
            "open_lots": pd.DataFrame(lots, columns=["lot_id", "symbol", "units", "signal_date",
                                                     "entry_date", "entry_price", "due_date"]),
            "diagnostics": pd.DataFrame(diagnostics, columns=["date", "symbol", "event"])}

## 5. Run the backtest with a momentum ranking

Rank stocks by their trailing 63-session return and run the strategy through 2013. This gives us scores we can inspect without fitting a model. Each purchase uses the previous market session's signal, including the signal just before the evaluation starts. Candidates must meet the quarterly membership and trailing-history requirements.

In [ ]:
scores = panel.loc[panel.eligible & panel.date.between('2012-12-31', '2013-12-31'),
                   ['date', 'symbol', 'momentum_63']].rename(columns={'momentum_63': 'score'})
result = run_backtest(prices, benchmark, scores, start='2013-01-01', end='2013-12-31')
history, fills = result['history'], result['fills']
display(fills.head(8))
display(history.head())
display(result['open_lots'])
print('Diagnostics:', len(result['diagnostics']))

In [ ]:
np.testing.assert_allclose(history.nav, history.cash + history.holdings_value)
cash_changes = fills.groupby('date').cash_change.sum().reindex(history.index, fill_value=0)
np.testing.assert_allclose(history.cash, 1_000_000 + cash_changes.cumsum(), atol=1e-7)
assert history.cash.min() >= -1e-7
assert fills.loc[fills.side.eq('BUY'), 'signal_date'].lt(fills.loc[fills.side.eq('BUY'), 'date']).all()
assert fills.loc[fills.side.eq('SELL'), 'date'].ge(fills.loc[fills.side.eq('SELL'), 'due_date']).all()
print('NAV, cash, signal timing, and maturity checks passed.')
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
history[['nav', 'benchmark_nav']].div(1_000_000).sub(1).plot(ax=axes[0])
axes[0].set_ylabel('Cumulative return')
history.cash.div(history.nav).plot(ax=axes[1], label='cash / NAV')
axes[1].set_ylabel('Cash fraction')
axes[1].legend()
plt.tight_layout()

Compare the cash balance with the NAV curve. Money waiting for another monthly cohort earns no interest here. A model can identify stocks with high future returns while the portfolio benefits only partly, because it has limited cash to buy them or buys them on only a few of the scored dates.

## 6. Measure performance from the portfolio's daily value

First calculate daily returns as today's NAV divided by yesterday's NAV, minus one. These returns include the effects of every open position, cash, and fees. They describe the portfolio we could hold, unlike an average of the model's hypothetical 90-day targets.

The `performance` function reports several views of that path:

- **Total return** compares final NAV with initial capital. **CAGR** converts that change to an equivalent compounded annual rate using elapsed calendar years; it is not an average of individual trade returns.
- **Maximum drawdown** measures the largest loss from a previous NAV peak. `nav.cummax()` gives the peak reached so far, and `nav / nav.cummax() - 1` gives the drawdown on each date. The initial capital row ensures the first entry fee counts as a loss too.
- **Annualized volatility** is the standard deviation of daily returns multiplied by $\sqrt{252}$.
- **Sharpe** compares mean daily excess return with daily volatility, then multiplies by $\sqrt{252}$. We subtract the daily equivalent of a 2% annual risk-free rate. This is a comparison hurdle; the portfolio's idle cash still earns zero in this model.

A high estimated Sharpe does not tell us how precisely it is measured, especially when returns are dependent over time. If daily volatility is zero, the ratio is undefined and the code reports it as missing.

In [ ]:
def performance(nav, annual_rf=0.02):
    """Daily simple returns, sample volatility, and a calendar-time CAGR."""
    nav = nav.sort_index()
    if len(nav) < 2 or nav.isna().any() or (nav <= 0).any():
        raise ValueError("Need at least two finite, positive NAV observations")
    if not np.isfinite(nav).all():
        raise ValueError("Nonfinite NAV")
    returns = nav.pct_change(fill_method=None).dropna()
    excess = returns - ((1 + annual_rf) ** (1 / 252) - 1)
    volatility = returns.std(ddof=1)
    years = (nav.index[-1] - nav.index[0]).days / 365.25
    return pd.Series({"final_value": nav.iloc[-1], "total_return": nav.iloc[-1] / nav.iloc[0] - 1,
                      "CAGR": (nav.iloc[-1] / nav.iloc[0]) ** (1 / years) - 1,
                      "max_drawdown": (nav / nav.cummax() - 1).min(),
                      "annualized_volatility": volatility * np.sqrt(252),
                      "Sharpe": np.sqrt(252) * excess.mean() / volatility if volatility > 1e-12 else np.nan})

In [ ]:
summary = pd.DataFrame({'strategy': performance(history.nav),
                        'S&P 500 total return': performance(history.benchmark_nav)})
display(summary)
print('Total fees:', history.fees.sum())
print('Gross traded / average NAV:', history.gross_traded.sum() / history.nav.mean())
print('Average cash fraction:', (history.cash / history.nav).mean())
drawdown = history[['nav', 'benchmark_nav']] / history[['nav', 'benchmark_nav']].cummax() - 1
drawdown.plot(title='Daily drawdown, including open lots')
plt.ylabel('Drawdown')
plt.tight_layout()

## Exercises

### Your turn: a different real lot and cost

Repeat the single-lot calculation for **MSFT**, signal close **2013-03-28**, a **$75,000 inclusive budget**, and **10 bp on each side**. Use the next market session for entry and the first quoted close on/after 90 calendar days for exit. Store `your_lot` as a dictionary with `entry_date`, `exit_date`, `units`, `entry_fee`, `exit_fee`, and `net_pnl`.

In [ ]:
your_lot = None

In [ ]:
if your_lot is None:
    print('The Friday after this signal is a market holiday; use the supplied calendar.')
else:
    msft = prices.loc[prices.symbol.eq('MSFT')].set_index('date').adj_close.sort_index()
    e = calendar[calendar.searchsorted(pd.Timestamp('2013-03-28'), side='right')]
    x = msft.index[msft.index.searchsorted(e + pd.Timedelta(days=90))]
    q = 75_000 / (msft.loc[e] * 1.001)
    assert your_lot['entry_date'] == e and your_lot['exit_date'] == x
    np.testing.assert_allclose([your_lot['units'], your_lot['entry_fee'], your_lot['exit_fee'], your_lot['net_pnl']],
                               [q, q*msft.loc[e]*0.001, q*msft.loc[x]*0.001, q*msft.loc[x]*0.999-75_000])
    print('Lot checks passed. Entry and exit costs use different notionals.')

### Your turn: compare the same policy at 0 and 25 bp

Run the 2013 momentum experiment twice without changing its scores or allocation rules. Store the outputs in `your_free_run` and `your_costly_run`. In `your_cost_note`, compare fees, final NAV, and cash. Explain why later unit quantities can differ even though rankings are unchanged.

In [ ]:
your_free_run = None
your_costly_run = None
your_cost_note = ''

In [ ]:
if your_free_run is None or your_costly_run is None:
    print('Vary only fee_bps; run each backtest from a clean initial state.')
else:
    assert np.isclose(your_free_run['history'].fees.sum(), 0)
    assert your_costly_run['history'].fees.sum() > 0
    for run in [your_free_run, your_costly_run]:
        h = run['history']
        np.testing.assert_allclose(h.nav, h.cash + h.holdings_value)
        assert h.cash.min() >= -1e-7
    assert your_cost_note.strip()
    print('Accounting checks passed. Compare paths as well as the final value.')

### Your turn: design accounting tests

Write a short test plan for a flat price with nonzero fees, a maturity on a holiday, two lots in the same stock, a missing entry quote, a temporary missing mark, an unresolved gap, insufficient cash, no orders, and an open lot at the final date. For each case, specify the expected cash balance, holding, or error. Small made-up price sequences help because you can calculate the correct answer by hand. Use cases like these to check your own backtester.